# CNN-ResBiGRU-SE for sEMG-based Hand Gesture Recognition
## NinaPro-DB5 Dataset

This notebook provides the complete implementation of the **CNN-ResBiGRU-SE** framework
for sEMG-based hand gesture recognition (HGR) using the **NinaPro-DB5** benchmark dataset.

---

### Architecture Overview
```
sEMG Input (window × channels)
        ↓
  [ConvB Block 1]  ← Conv1D-BN-ReLU × 3 (strided)
        ↓
  [ConvB Block 2]  ← Conv1D-BN-ReLU × 3 (strided)
        ↓
  [ResBiGRU Block] ← BiGRU + LayerNorm + Residual connection
        ↓
  [SE Block]       ← Squeeze-and-Excitation (channel attention)
        ↓
  [GAP + Dense]    ← Global Average Pooling + SoftMax
        ↓
  Gesture Label
```

### Dataset: NinaPro-DB5
| Property | Value |
|---|---|
| Subjects | 10 non-disabled |
| Channels | 16 (dual Thalmic Myo armbands, 8 channels each) |
| Gestures | 52 (Exercise A: 12, B: 17, C: 23) + rest |
| Repetitions | 6 per gesture |
| Sampling Rate | 200 Hz |

> **Note on dual-armband configuration:** One Myo armband is placed near the elbow
> at the radiohumeral joint, the second is rotated 22.5° toward the wrist. This
> arrangement provides broader forearm muscle coverage without arm shaving.

### Reference
> S. Mekruksavanich, N. Hnoohom, and A. Jitpattanakul, *"Deep Residual Network with
> Channel Attention for Improving Hand Gesture Recognition with Surface Electromyography
> Signal,"* IEEE Access, 2024.


In [ ]:
# ─── Cell 1: Install dependencies ────────────────────────────────────────────
!pip install scipy scikit-learn seaborn tensorflow --quiet
print('Dependencies ready.')


In [ ]:
# ─── Cell 2: Imports ─────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import butter, filtfilt, iirnotch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers

tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')


In [ ]:
# ─── Cell 3: Configuration ────────────────────────────────────────────────────
# ═══════════════════════════════════════════════════════════════════════════════

# ── Dataset settings ──
NUM_SUBJECTS   = 10
NUM_CHANNELS   = 16        # 2 × Myo armband (8 channels each)
SAMPLING_RATE  = 200       # Hz
EXERCISE_SIZES = {'E1': 12, 'E2': 17, 'E3': 23}

# ── Experiment: select exercise subset ──
# Options: 'E1' | 'E2' | 'E3' | 'E1+E2+E3'
EXERCISE = 'E1+E2+E3'      # Full 52-gesture set

# ── Windowing (200 ms window / 50 ms stride at 200 Hz) ──
WINDOW_SIZE   = 40         # samples  (200 ms × 200 Hz)
WINDOW_STRIDE = 10         # samples  ( 50 ms × 200 Hz)

# ── Pre-processing parameters for NinaPro-DB5 (raw sEMG at 200 Hz) ──
# Nyquist limit = 100 Hz  →  upper cutoff must be < 100 Hz
BP_LOWCUT   = 20.0         # Hz  (bandpass low  cut)
BP_HIGHCUT  = 90.0         # Hz  (bandpass high cut, below Nyquist of 100 Hz)
BP_ORDER    = 4
NOTCH_FREQ  = 50.0         # Hz  (power-line interference)
NOTCH_Q     = 30.0         # Quality factor

# ── Model hyperparameters ──
CONV_FILTERS_1  = 64
CONV_FILTERS_2  = 128
GRU_UNITS       = 128
SE_REDUCTION    = 16
L2_LAMBDA       = 1e-4

# ── Training hyperparameters (Table 2 in manuscript) ──
N_FOLDS        = 5
N_EPOCHS       = 200
BATCH_SIZE     = 32
LEARNING_RATE  = 0.001
BETA_1, BETA_2 = 0.9, 0.999
EPSILON        = 1e-8
PATIENCE       = 20

OUTPUT_DIR = '/content/results_db5'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Configuration loaded.')


In [ ]:
# ─── Cell 4: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# !! UPDATE THIS PATH to where you placed the NinaPro-DB5 folder on your Drive !!
# Expected structure:
#   DB5_PATH/
#     S1_E1_A1.mat   S1_E2_A1.mat   S1_E3_A1.mat
#     S2_E1_A1.mat   ...
#     ...            S10_E3_A1.mat
DB5_PATH = '/content/drive/MyDrive/NinaPro/DB5'   # <── update this

if not os.path.isdir(DB5_PATH):
    raise FileNotFoundError(
        f'Dataset not found at: {DB5_PATH}\n'
        'Download NinaPro-DB5 from https://ninapro.hevs.ch/instructions/DB5.html\n'
        'and update DB5_PATH above.'
    )
print(f'Dataset path OK: {DB5_PATH}')


In [ ]:
# ─── Cell 5: Data Loading ─────────────────────────────────────────────────────
def load_subject_db5(data_path, subject_id, exercises):
    """
    Load NinaPro-DB5 .mat files for one subject.

    NinaPro-DB5 uses dual Thalmic Myo armbands (8 channels each = 16 total).
    File naming convention: S{id}_E{ex_num}_A1.mat

    Args:
        data_path  : root directory of DB5
        subject_id : integer 1..10
        exercises  : list such as ['E1','E2','E3']

    Returns:
        emg         : (N, 16) raw differential sEMG, float32
        labels      : (N,)    gesture labels (0 = rest)
        repetitions : (N,)    repetition numbers
    """
    emg_all, lbl_all, rep_all = [], [], []
    label_offset = 0

    for ex in exercises:
        ex_num   = {'E1': 1, 'E2': 2, 'E3': 3}[ex]
        filename = f'S{subject_id}_E{ex_num}_A1.mat'

        for candidate in [
            os.path.join(data_path, filename),
            os.path.join(data_path, f'S{subject_id}', filename)
        ]:
            if os.path.isfile(candidate):
                filepath = candidate
                break
        else:
            raise FileNotFoundError(f'Cannot find {filename} in {data_path}')

        mat    = sio.loadmat(filepath)
        emg    = mat['emg'].astype(np.float32)           # (N, 16)
        labels = mat['restimulus'].flatten().astype(int)  # (N,)
        reps   = mat['rerepetition'].flatten().astype(int)# (N,)

        non_rest = labels > 0
        labels[non_rest] += label_offset
        label_offset += EXERCISE_SIZES[ex]

        emg_all.append(emg)
        lbl_all.append(labels)
        rep_all.append(reps)

    return (
        np.concatenate(emg_all,  axis=0),
        np.concatenate(lbl_all,  axis=0),
        np.concatenate(rep_all,  axis=0)
    )

print('load_subject_db5() defined.')


In [ ]:
# ─── Cell 6: Preprocessing ───────────────────────────────────────────────────
def butter_bandpass(lowcut, highcut, fs, order=4):
    """Design a Butterworth bandpass filter."""
    nyq = 0.5 * fs
    assert highcut < nyq, (
        f'highcut ({highcut} Hz) must be below Nyquist ({nyq} Hz) '
        f'for fs={fs} Hz.')
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype='band')
    return b, a


def notch_filter(freq, fs, Q=30.0):
    """Design an IIR notch filter."""
    b, a = iirnotch(freq / (0.5 * fs), Q)
    return b, a


def preprocess_emg_db5(emg, fs=200,
                        bp_low=20.0, bp_high=90.0, bp_order=4,
                        notch_freq=50.0, notch_Q=30.0):
    """
    Preprocessing pipeline for NinaPro-DB5 (Myo armbands, 200 Hz).

    Pipeline (applied per channel):
        1. 4th-order Butterworth bandpass  20 – 90 Hz
           (upper cutoff is kept below Nyquist = 100 Hz)
        2. IIR notch filter at 50 Hz  (power-line interference)
        3. Zero-mean, unit-variance normalisation

    Args:
        emg      : (N, 16) raw differential sEMG, float32
        fs       : sampling frequency in Hz
        bp_low   : bandpass lower cutoff (Hz)
        bp_high  : bandpass upper cutoff (Hz)  — must be < fs/2
        bp_order : Butterworth filter order
        notch_freq: notch centre frequency (Hz)
        notch_Q  : notch quality factor

    Returns:
        emg_out  : (N, 16) filtered and normalised sEMG, float32
    """
    emg_f = emg.copy()

    # 1. Bandpass
    b_bp, a_bp = butter_bandpass(bp_low, bp_high, fs, bp_order)
    # 2. Notch
    b_n,  a_n  = notch_filter(notch_freq, fs, notch_Q)

    for ch in range(emg_f.shape[1]):
        sig        = emg_f[:, ch].astype(np.float64)
        sig        = filtfilt(b_bp, a_bp, sig)   # bandpass
        sig        = filtfilt(b_n,  a_n,  sig)   # notch
        emg_f[:, ch] = sig

    # 3. Normalisation
    mu  = emg_f.mean(axis=0, keepdims=True)
    std = emg_f.std(axis=0,  keepdims=True) + 1e-8
    return ((emg_f - mu) / std).astype(np.float32)


def segment_windows(emg, labels, repetitions, window_size, stride):
    """
    Segment sEMG into fixed-length windows.
    Only windows with a single, unique, non-rest gesture label are kept.
    """
    X_list, y_list, r_list = [], [], []
    for start in range(0, len(emg) - window_size + 1, stride):
        end    = start + window_size
        seg    = labels[start:end]
        active = seg[seg > 0]
        unique = np.unique(active)
        if len(unique) == 1:
            X_list.append(emg[start:end])
            y_list.append(int(unique[0]) - 1)
            r_list.append(int(repetitions[start + window_size // 2]))
    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.int32),
        np.array(r_list, dtype=np.int32)
    )

print('Preprocessing functions defined.')


In [ ]:
# ─── Cell 7: CNN-ResBiGRU-SE Architecture ────────────────────────────────────
# (Identical architecture for both DB1 and DB5; only input_shape differs)

def conv_block(x, num_filters, kernel_size=3, use_stride=True, l2=1e-4):
    """
    Convolutional Block: [Conv1D → BN → ReLU] × 3
    Third Conv1D uses stride=2 to halve temporal resolution.
    """
    for s in [1, 1, (2 if use_stride else 1)]:
        x = layers.Conv1D(
                num_filters, kernel_size,
                strides=s, padding='same',
                kernel_regularizer=regularizers.l2(l2),
                use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x


def resbigru_block(x, gru_units, l2=1e-4):
    """
    Residual Bidirectional GRU Block (Eq. 3, manuscript).
    Forward & backward GRU hidden states are concatenated;
    a residual connection and LayerNorm are applied.
    """
    bigru_out = layers.Bidirectional(
        layers.GRU(gru_units, return_sequences=True,
                   kernel_regularizer=regularizers.l2(l2)),
        name='bigru')(x)

    if x.shape[-1] != bigru_out.shape[-1]:
        residual = layers.Dense(bigru_out.shape[-1],
                                use_bias=False, name='res_proj')(x)
    else:
        residual = x

    # LN: x_hat_i = (x_i - mu) / sqrt(var + eps) * gamma + beta
    return layers.LayerNormalization(epsilon=1e-8,
                                     name='resbigru_ln')(residual + bigru_out)


def se_block(x, reduction_ratio=16):
    """
    Squeeze-and-Excitation Block (Eq. 4 & 5, manuscript).
    Squeeze: Z_c = mean_t(U_c(t))  [Global Average Pooling over time]
    Excitation: s = sigmoid(W2 * ReLU(W1 * z))
    """
    C = x.shape[-1]
    r = max(1, C // reduction_ratio)
    z = layers.GlobalAveragePooling1D(name='se_gap')(x)
    s = layers.Dense(r, activation='relu',    name='se_fc1')(z)
    s = layers.Dense(C, activation='sigmoid', name='se_fc2')(s)
    s = layers.Reshape((1, C), name='se_reshape')(s)
    return layers.Multiply(name='se_scale')([x, s])


def build_cnn_resbigru_se(input_shape, num_classes,
                           conv_f1=64, conv_f2=128,
                           gru_units=128, se_r=16, l2=1e-4):
    """
    CNN-ResBiGRU-SE model.
    For DB5:  input_shape = (40, 16)  → window=40 samples, 16 channels
    For DB1:  input_shape = (20, 10)  → window=20 samples, 10 channels
    """
    inp = keras.Input(shape=input_shape, name='sEMG_input')

    x = conv_block(inp, conv_f1, kernel_size=3, use_stride=True, l2=l2)
    x = conv_block(x,   conv_f2, kernel_size=3, use_stride=True, l2=l2)
    x = resbigru_block(x, gru_units, l2=l2)
    x = se_block(x, reduction_ratio=se_r)
    x = layers.GlobalAveragePooling1D(name='gap')(x)
    out = layers.Dense(num_classes, activation='softmax',
                       name='gesture_output')(x)

    return Model(inputs=inp, outputs=out, name='CNN_ResBiGRU_SE')


# Sanity check
demo = build_cnn_resbigru_se(
    input_shape=(WINDOW_SIZE, NUM_CHANNELS), num_classes=52)
demo.summary()


In [ ]:
# ─── Cell 8: Training Function ────────────────────────────────────────────────
def train_subject_kfold(X, y, reps, num_classes,
                         n_folds=5, n_epochs=200, batch_size=32):
    """
    Train CNN-ResBiGRU-SE for one subject using k-fold CV.

    Folds are defined at the *repetition* level (leakage-free):
      - NinaPro-DB5 has 6 repetitions per subject.
      - 5-fold CV assigns repetitions to folds; windowing is applied
        independently within each train/test partition.
    """
    unique_reps = np.unique(reps)
    kf          = KFold(n_splits=n_folds, shuffle=False)

    fold_accs, fold_f1s = [], []
    all_true, all_pred  = [], []

    for fold, (tr_idx, te_idx) in enumerate(kf.split(unique_reps)):
        train_reps = unique_reps[tr_idx]
        test_reps  = unique_reps[te_idx]

        tr_mask = np.isin(reps, train_reps)
        te_mask = np.isin(reps, test_reps)

        X_tr, y_tr = X[tr_mask], y[tr_mask]
        X_te, y_te = X[te_mask], y[te_mask]

        y_tr_cat = tf.keras.utils.to_categorical(y_tr, num_classes)
        y_te_cat = tf.keras.utils.to_categorical(y_te, num_classes)

        model = build_cnn_resbigru_se(
            input_shape=X_tr.shape[1:],
            num_classes=num_classes,
            conv_f1=CONV_FILTERS_1, conv_f2=CONV_FILTERS_2,
            gru_units=GRU_UNITS, se_r=SE_REDUCTION, l2=L2_LAMBDA
        )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=LEARNING_RATE,
                beta_1=BETA_1, beta_2=BETA_2, epsilon=EPSILON
            ),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        model.fit(
            X_tr, y_tr_cat,
            validation_data=(X_te, y_te_cat),
            epochs=n_epochs, batch_size=batch_size,
            callbacks=[
                tf.keras.callbacks.EarlyStopping(
                    monitor='val_accuracy', patience=PATIENCE,
                    restore_best_weights=True, verbose=0),
                tf.keras.callbacks.ReduceLROnPlateau(
                    monitor='val_loss', factor=0.5, patience=10,
                    min_lr=1e-6, verbose=0)
            ],
            verbose=0
        )

        y_pred = np.argmax(model.predict(X_te, verbose=0), axis=1)
        acc    = accuracy_score(y_te, y_pred) * 100
        f1     = f1_score(y_te, y_pred, average='macro', zero_division=0) * 100

        fold_accs.append(acc);  fold_f1s.append(f1)
        all_true.extend(y_te.tolist())
        all_pred.extend(y_pred.tolist())

        print(f'    Fold {fold+1}/{n_folds}  Acc={acc:.2f}%  F1={f1:.2f}%')

    return np.mean(fold_accs), np.mean(fold_f1s), all_true, all_pred

print('Training function defined.')


In [ ]:
# ─── Cell 9: Main Training Loop ───────────────────────────────────────────────
if EXERCISE == 'E1+E2+E3':
    ex_list, num_classes = ['E1', 'E2', 'E3'], 52
elif EXERCISE == 'E1':
    ex_list, num_classes = ['E1'], 12
elif EXERCISE == 'E2':
    ex_list, num_classes = ['E2'], 17
elif EXERCISE == 'E3':
    ex_list, num_classes = ['E3'], 23
else:
    raise ValueError(f'Unknown EXERCISE: {EXERCISE}')

print(f'Dataset      : NinaPro-DB5')
print(f'Exercise     : {EXERCISE}  ({num_classes} classes)')
print(f'Window/Stride: {WINDOW_SIZE}/{WINDOW_STRIDE} samples '
      f'({WINDOW_SIZE/SAMPLING_RATE*1000:.0f}/{WINDOW_STRIDE/SAMPLING_RATE*1000:.0f} ms)')
print('='*60)

subject_accs, subject_f1s, all_results = [], [], []

for subj in range(1, NUM_SUBJECTS + 1):
    print(f'\nSubject {subj:02d}/{NUM_SUBJECTS}')

    emg, labels, repetitions = load_subject_db5(DB5_PATH, subj, ex_list)

    # Preprocessing: bandpass (20-90 Hz) + 50 Hz notch + normalisation
    emg_norm = preprocess_emg_db5(
        emg, fs=SAMPLING_RATE,
        bp_low=BP_LOWCUT, bp_high=BP_HIGHCUT, bp_order=BP_ORDER,
        notch_freq=NOTCH_FREQ, notch_Q=NOTCH_Q
    )

    X, y, reps = segment_windows(
        emg_norm, labels, repetitions, WINDOW_SIZE, WINDOW_STRIDE)

    if len(X) == 0:
        print(f'  [WARNING] No valid windows — skipped.')
        continue

    print(f'  Windows  : {X.shape}   Classes: {np.unique(y).size}')

    acc, f1, y_true, y_pred = train_subject_kfold(
        X, y, reps, num_classes,
        n_folds=N_FOLDS, n_epochs=N_EPOCHS, batch_size=BATCH_SIZE
    )

    subject_accs.append(acc);  subject_f1s.append(f1)
    all_results.append({'Subject': subj, 'Accuracy': acc, 'F1_Score': f1})
    print(f'  ► Subject {subj:02d}  Acc={acc:.2f}%  F1={f1:.2f}%')

mean_acc = np.mean(subject_accs)
mean_f1  = np.mean(subject_f1s)
print('\n' + '='*60)
print(f'OVERALL ({EXERCISE})')
print(f'  Avg Accuracy : {mean_acc:.2f}% ± {np.std(subject_accs):.2f}%')
print(f'  Avg F1-Score : {mean_f1:.2f}% ± {np.std(subject_f1s):.2f}%')
print('='*60)

csv_path = os.path.join(OUTPUT_DIR, f'results_{EXERCISE}.csv')
pd.DataFrame(all_results).to_csv(csv_path, index=False)
print(f'Results saved → {csv_path}')


In [ ]:
# ─── Cell 10: Results Visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'CNN-ResBiGRU-SE  |  NinaPro-DB5  |  {EXERCISE}', fontsize=13, fontweight='bold')

axes[0].boxplot(
    subject_accs, patch_artist=True, showmeans=True,
    boxprops=dict(facecolor='#A9DFBF'),
    medianprops=dict(color='darkgreen', linewidth=2),
    meanprops=dict(marker='x', markeredgecolor='red', markersize=10)
)
axes[0].axhline(mean_acc, color='red', linestyle='--',
                linewidth=1.5, label=f'Mean = {mean_acc:.2f}%')
axes[0].set_title('Accuracy Distribution (all subjects)')
axes[0].set_ylabel('Accuracy (%)'); axes[0].set_xlabel('All Subjects')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

colors = ['#1A5276' if a >= mean_acc else '#C0392B' for a in subject_accs]
axes[1].bar(range(1, len(subject_accs)+1), subject_accs,
            color=colors, edgecolor='k', linewidth=0.5)
axes[1].axhline(mean_acc, color='red', linestyle='--',
                linewidth=1.5, label=f'Mean = {mean_acc:.2f}%')
axes[1].set_title('Per-Subject Recognition Accuracy')
axes[1].set_xlabel('Subject ID'); axes[1].set_ylabel('Accuracy (%)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(OUTPUT_DIR, f'accuracy_{EXERCISE}.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')
